In [12]:
import os
import pandas as pd
import subprocess as sp
import polars as pl

In [13]:
out_dir = "../../results/04_single_cell_access_atac/26_split_fragment"
os.makedirs(out_dir, exist_ok=True)

In [14]:
df_meta = pd.read_csv("../../results/04_single_cell_access_atac/25_annotation_by_marker_score/obj_metadata.csv", index_col=0)

In [15]:
df_meta.head()

,orig.ident,nCount_ATAC,nFeature_ATAC,Sample,TSSEnrichment,ReadsInTSS,ReadsInPromoter,PromoterRatio,PassQC,NucleosomeRatio,...,Ciliated_Marker_Score_1,Tuft_Marker_Score_1,Neuroendocrine_Marker_Score_1,Ionocyte_Marker_Score_1,Goblet_Marker_Score_1,Hillock_Marker_Score_1,Hillock_luminal_Marker_Score_1,Hillock2_Marker_Score_1,cell_type_final,cell_type_final_v2
mouse#GAGGCTCCAGAGTCGA,SeuratProject,33439,33439,mouse,10.229,10326,33420,0.142007,1,4.658300,...,-0.004259,-0.017897,0.065168,0.116702,-0.116437,0.757399,0.773869,0.345944,Hillock,Basal
mouse#TATCGAGGTAACGTAA,SeuratProject,35117,35117,mouse,10.541,10338,33123,0.143647,1,3.778984,...,-0.013413,-0.036434,0.036397,0.018903,-0.023247,0.566581,0.334563,-0.098547,Basal,Basal
mouse#CGCAGGTAGAACGTCG,SeuratProject,30979,30979,mouse,9.835,8950,29135,0.135571,1,4.227330,...,-0.015304,-0.023902,0.069732,0.035492,-0.040106,0.632139,0.355674,-0.090492,Basal,Basal
mouse#GCTTGCTGTTGTGAGG,SeuratProject,32349,32349,mouse,11.220,9678,30277,0.150201,1,4.268308,...,-0.013825,-0.011471,0.023511,0.081332,-0.034057,0.518282,0.296790,-0.135756,Basal,Basal
mouse#TCCGACTAGAACGTTA,SeuratProject,29864,29864,mouse,11.576,10242,31775,0.157905,1,3.592779,...,0.080804,1.015369,0.047278,0.259541,-0.108228,0.220731,0.008195,-0.000901,Tuft/Neuroendocrine,Tuft


In [16]:
df_meta["barcode"] = df_meta.index
df_meta["barcode"] = df_meta["barcode"].str.split("#").str[1]
df_meta = df_meta[["barcode", "cell_type_final_v2"]]

In [17]:
df_meta.head()

,barcode,cell_type_final_v2
mouse#GAGGCTCCAGAGTCGA,GAGGCTCCAGAGTCGA,Basal
mouse#TATCGAGGTAACGTAA,TATCGAGGTAACGTAA,Basal
mouse#CGCAGGTAGAACGTCG,CGCAGGTAGAACGTCG,Basal
mouse#GCTTGCTGTTGTGAGG,GCTTGCTGTTGTGAGG,Basal
mouse#TCCGACTAGAACGTTA,TCCGACTAGAACGTTA,Tuft


In [18]:
df_fragments = pl.read_csv(
            "../../results/04_single_cell_access_atac/09_merge_fragment/merged.fragments.aggregated.tsv.gz",
            skip_rows=0,
            has_header=False,
            separator="\t",
            use_pyarrow=False,
            new_columns=["Chromosome", "Start", "End", "Barcode", "Count", "Edit"])

df_fragments = df_fragments.drop("column_7")

In [19]:
df_fragments.head()

Chromosome,Start,End,Barcode,Count,Edit
str,i64,i64,str,i64,str
"""chr1""",3050003,3050371,"""GGATAGGAGTGCTAGG""",16,"""3050117|3050122|3050123|305012…"
"""chr1""",3050004,3050131,"""TTCATCAAGAACCCGA""",14,"""3050099|3050114|3050130"""
"""chr1""",3050004,3050149,"""TTACGTAAGCACCATA""",1,"""3050116|3050117|3050130|305013…"
"""chr1""",3050004,3050149,"""TTACGTTAGAACCATA""",12,"""3050116|3050117|3050130|305013…"
"""chr1""",3050004,3050149,"""TTACGTTAGCACCATT""",1,"""3050071|3050116|3050117|305013…"


In [20]:
df_fragments = df_fragments.filter(pl.col("Chromosome").is_in(["chr1","chr2","chr3","chr4","chr5","chr6","chr7","chr8","chr9","chr10",
                                                              "chr11","chr12","chr13","chr14","chr15","chr16","chr17","chr18","chr19",
                                                              "chrX","chrY"]))

In [21]:
df_fragments.head()

Chromosome,Start,End,Barcode,Count,Edit
str,i64,i64,str,i64,str
"""chr1""",3050003,3050371,"""GGATAGGAGTGCTAGG""",16,"""3050117|3050122|3050123|305012…"
"""chr1""",3050004,3050131,"""TTCATCAAGAACCCGA""",14,"""3050099|3050114|3050130"""
"""chr1""",3050004,3050149,"""TTACGTAAGCACCATA""",1,"""3050116|3050117|3050130|305013…"
"""chr1""",3050004,3050149,"""TTACGTTAGAACCATA""",12,"""3050116|3050117|3050130|305013…"
"""chr1""",3050004,3050149,"""TTACGTTAGCACCATT""",1,"""3050071|3050116|3050117|305013…"


In [ ]:
for cell_type in df_meta["cell_type_final_v2"].unique():
    selected_barcodes = df_meta[df_meta["cell_type_final_v2"] == cell_type]["barcode"].tolist()

    df_filtered_fragments = df_fragments.filter(pl.col("Barcode").is_in(selected_barcodes))
    
    output_path = os.path.join(out_dir, f"{cell_type}.fragments.tsv")
    df_filtered_fragments.write_csv(output_path, separator="\t", include_header=False)
    sp.run(["bgzip", "-f", f"{out_dir}/{cell_type}.fragments.tsv"])